# Lab-3 Data Clearning

## Establishe connection to Spark

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Lab-3-DataClearning") \
    .config("spark.ui.port", "4040") \
    .config("spark.ui.enabled", "true") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "lab3/lakehouse") 

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark version: {spark.version} with Delta support Lab-3")

Spark version: 3.5.0 with Delta support Lab-3


##  1. Generate dirty data

In [5]:
from faker import Faker
import pandas as pd
import random

fake = Faker()

def generate_dirty_data(n):
    data = []
    for i in range(n):
        data.append({
            "transaction_id": i if i > 5 else 1, # Створюємо навмисні дублікати для перших 5 записів
            "client_name": fake.name(),
            "email": fake.email() if random.random() > 0.1 else None, # 10% пропущених email
            "amount": random.uniform(-100, 1000), # Від'ємні суми
            "transaction_date": fake.date_time_this_year()
        })
    return pd.DataFrame(data)

dirty_df = spark.createDataFrame(generate_dirty_data(10000))
dirty_df.createOrReplaceTempView("raw_transactions")

## 2. Data clarning (Silver Layer)

In [6]:
# Clearning using Spark API
cleaned_df = dirty_df \
    .dropDuplicates(["transaction_id"]) \
    .filter("amount > 0") \
    .fillna({"email": "unknown@example.com"})

# Check result
print(f"Records before: {dirty_df.count()}")
print(f"Records after clearning: {cleaned_df.count()}")

cleaned_df.write.format("delta").mode("overwrite").saveAsTable("silver_transactions")

Records before: 10000
Records after clearning: 9120
